<a href="https://colab.research.google.com/github/NguyenBaTam-tristan/deepml-curriculum-scraper/blob/main/Designed_curriculum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#!/usr/bin/env python3
"""
Sinh lịch học Deep-ML 07/10/2026 -> 31/12/2026 (Mon-Sat học, Sun nghỉ).
Dùng:  python build_schedule.py deepml_v2_20261006_043044.json learning_schedule_deepml_2026.csv
"""
import csv, json, re, sys
from datetime import date, timedelta

SRC = "deepml_v2_20261006_043044.json"
OUT = "learning_schedule_deepml_2026.csv"
START, END = date(2026, 10, 7), date(2026, 12, 31)

# ---------- ước lượng thời gian (giờ / bài) ----------
H = {"Easy": 0.6, "Medium": 1.1, "Hard": 2.4, "math": 0.4}
DAY_MIN, DAY_MAX = 3.5, 4.0
RANK = {"Easy": 0, "Medium": 1, "Hard": 2}

# ---------- 51 bài đã làm (khớp theo tên) ----------
DONE_PROBLEMS = """Calculate 2x2 Matrix Inverse|Calculate Accuracy Score|Calculate Cosine Similarity Between Vectors|
Calculate F1 Score from Predicted and True Labels|Calculate Mean by Row or Column|Compute the Cross Product of Two 3D Vectors|
Convert Vector to Diagonal Matrix|Derivative of a Polynomial|Descriptive Statistics Calculator|Dot Product Calculator|
Empirical Probability Mass Function (PMF)|Expected Value and Variance of an n-Sided Die|Feature Scaling Implementation|
Gradient Direction and Magnitude|Linear Regression Using Gradient Descent|Linear Regression Using Normal Equation|
Matrix Determinant & Trace|Matrix-Vector Dot Product|Min-Max Scaling of Feature Values|One-Hot Encoding of Nominal Values|
Reshape Matrix|Scalar Multiplication of a Matrix|Sigmoid Activation Function Understanding|Softmax Activation Function Implementation|
Transformation Matrix from Basis B to C|Transpose of a Matrix|Vector Element-wise Sum|Vector Norms (L1/L2/L-inf) and the Frobenius Norm|
Calculate Eigenvalues of a Matrix|Implement K-Fold Cross-Validation|Implement K-Nearest Neighbors|Matrix times Matrix|Matrix Transformation|
Quotient Rule for Derivatives""".replace("\n", "").split("|")
DONE_MATH = """Derivatives and Gradients|Descriptive Statistics|Expectation and Variance Algebra|Gradient Descent Updates|Matrix Basics|
ML Workflow Basics|Probability Fundamentals|Vector Operations|Covariance and Correlation|Determinants and Trace|Inverse and Rank|
Least Squares and the Normal Equations|Matrix Multiplication|Orthogonality and Projections|Solving Linear Systems|
The Four Fundamental Subspaces|Vector Norms and Linear Independence""".replace("\n", "").split("|")
norm = lambda s: re.sub(r"\s+", " ", s.strip().lower())

# ---------- chủ đề (thứ tự học) ----------
THEMES = {
    1: ("Linear Algebra & Calculus", "Nền toán để suy ra gradient/ma trận cho mọi layer và hiểu low-rank, SVD dùng trong LoRA/nén mô hình."),
    2: ("Probability & Information Theory", "Hiểu loss (cross-entropy, KL, MLE) và perplexity - thước đo chất lượng khi quantize/nén LLM."),
    3: ("Autograd, Backprop & Optimizers", "Tự viết backprop/optimizer/scheduler là nền của phong cách micrograd/nanoGPT."),
    4: ("Deep Learning Building Blocks", "Chuẩn hóa, khởi tạo, embedding, tokenizer, training loop - các khối nhỏ của một LLM."),
    5: ("Attention & Transformer", "Làm chủ attention (MHA/MQA/GQA/MLA, RoPE, mask) - lõi tính toán và bộ nhớ của LLM."),
    6: ("LLM Inference & KV-Cache", "Hiểu KV-cache, PagedAttention, FlashAttention, batching, roofline - nút thắt thật sự khi serve LLM."),
    7: ("Quantization, LoRA & MoE", "Nén mô hình (INT8/FP8/FP4), LoRA, MoE để chạy LLM vừa bộ nhớ/ngân sách trên thiết bị biên."),
    8: ("Systems, Kernels & Serving", "Tư duy hệ thống: kernel, memory hierarchy, parallelism, MLOps, đo latency/cost khi triển khai."),
    9: ("RLHF & Alignment", "DPO/PPO/GRPO - hiểu bước hậu huấn luyện để tinh chỉnh LLM nhỏ cho edge."),
    10: ("Extended Math (ưu tiên thấp)", "Mở rộng nền tảng thống kê/ML/GNN/diffusion; làm khi đã xong lõi, có thể trượt sang 2027."),
}

# math: luật khớp theo thứ tự ưu tiên (cụ thể -> chung)
MATH_RULES = [
    (9, r"dpo|bradley|ppo|policy gradient|advantage|grpo|bellman|importance sampling|discounted return|td error|goodhart|bandit|thompson|insight|learning barrier|entropy collapse|sequential attempts|internaliz|one update|supervision from solution|deconfounded|symmetries, parallel|sound deduction|deduction versus|galois|lattices"),
    (6, r"kv cache|bytes in the kv|arithmetic intensity|roofline|speculative|online softmax|pagedattention|prefix-cache|quantized kv|memory hierarchy|disaggregation|decode latency"),
    (7, r"quantiz|floating-point|lora|low-rank"),
    (5, r"attention|vit patch|dictionary lookup"),
    (8, r"serving|cold start|cost per million|little's|tail latency|ring all|tensor parallelism|kernel fusion|memory traffic|register blocking|gpu mapping|strides|floor division|interval bounds|loop nests|term rewriting|movement ops|federated|gaussian mechanism|conformal|fused"),
    (3, r"backprop|chain rule|neural network deriv|adam|minibatch|optimization: convex|gradient descent|softmax and cross|variance preservation|perplexity"),
    (4, r"convolution|recurrent|lstm|forget gate|constant error|error flow|truncated|word2vec|perceptron|regularization and generalization|interpolation|double descent"),
    (2, r"probab|bayes|distribution|gaussian|kl divergence|entropy|likelihood|large numbers|inference|calibration|expectation|information|maximum|ml workflow|bias.variance|log-likelihood"),
    (1, r"^matrix calculus|^eigen|^matrix decomp|^gram|^pseudoinverse|^positive definite|^multivariate calculus|^taylor|^fourier|^effective rank|^derivatives|^matrix|^vector|^solving|^inverse|^determinants|^orthogonal|^least squares|^the four|^pca via"),
]

# problems: nhóm bắt buộc + từ khoá cốt lõi
CORE_KW = r"autograd|backprop|backward|adam|sgd|muon|momentum|optimizer|scheduler|schedule|warmup|clipping|rmsnorm|layer ?norm|batch ?norm|normalization|attention|mqa|gqa|mla|paged|flash|rope|rotary|kv.?cache|quantiz|int8|fp8|fp4|mxfp4|continuous batching|roofline|speculative|tokeniz|bpe|embedding|cross-entropy|softmax|lora|moe|mixture of experts|dpo|ppo|grpo|rlhf|reward model|perplexity|sampling|decod|prefill|checkpoint|mixed precision|all-reduce|ddp|parallel|mfu|latency|throughput|bandwidth|fusion|kernel|memory|gpt|transformer|residual|dropout|initialization|distillation|pruning"
NEG_KW = r"video|diffusion|\bgan\b|vae|vq-|mdn|thanksgiving|weaver|ddim|ddpm|rectified|world model|trajectory|latent action|agentopsd|insight|lattice|graph|gcn|alphazero|a3c|maskgit|tdnn|hinge|volume bars|dollar bars|power grid|t-sne|elu\b|softplus|softsign|selu|hardtanh|hard sigmoid|mish|square relu|adamax|adadelta|local response|asr|tts|voice|audio|speech|recording|procurement|blackjack|gambl|mountain|cliff|atari|frozen|taxi|gridworld"
MANDATORY = {"Inference", "Optimization", "MLOps", "Performance Optimization", "Systems Programming"}
EXTRA_CATS = {"Deep Learning", "pytorch", "cuda", "triton", "Compilers", "tinygrad", "NLP", "LLM", "Linear Algebra", "Machine Learning", "ML", "Machine Learning Systems"}
RL_OK = r"ppo|dpo|grpo|rlhf|reward model|policy gradient with|rloo|gae|preference"

P_RULES = [
    (9, r"dpo|ppo|grpo|rlhf|reward|rloo|gae|preference|policy gradient"),
    (7, r"quantiz|int8|fp8|fp4|mxfp4|lora|moe|mixture of experts|expert|pruning|distillation|sparsity|mini-float|mixed precision|number format"),
    (6, r"kv.?cache|paged|flash|continuous batching|speculative|prefill|decode|decoding|sampling|roofline|arithmetic intensity|tiled attention|prefix|beam|latency|throughput|goodput|fusion|memory traffic|bandwidth|radix|copy-on-write|fragmentation|preemption"),
    (8, r"cuda|triton|compiler|mlops|kernel|parallel|all-reduce|ddp|mfu|autoscal|serving|deploy|monitor|drift|pipeline|tinygrad|tensor core|profil|performance"),
    (5, r"attention|mqa|gqa|mla|rope|rotary|positional|transformer|causal|mask|gpt"),
    (4, r"norm|dropout|embedding|tokeniz|bpe|initialization|residual|pooling|convolution|lstm|rnn|gru|weight tying|fine-?tuning|training loop|dataloader|collate"),
    (3, r"autograd|backprop|backward|gradient|adam|sgd|muon|momentum|optimizer|scheduler|schedule|warmup|clipping|loss|cross-entropy|softmax|neuron|derivative"),
    (2, r"probab|bayes|entropy|kl |likelihood|gaussian|expectation|information|convex|optimization"),
    (1, r"matrix|eigen|svd|determinant|inverse|vector|projection|decomposition|qr|lu |orthogonal|tensor|jacobian|hessian|taylor|derivative"),
]


def theme_of(title, rules, default):
    t = title.lower()
    for th, pat in rules:
        if re.search(pat, t):
            return th
    return default


def prio(x):
    t, c = x["title"].lower(), x["category"]
    s = 0.0
    if c in MANDATORY:
        s += 6
    if re.search(CORE_KW, t):
        s += 5
    if c in ("Deep Learning", "Inference"):
        s += 2
    if c == "Linear Algebra" and x["difficulty"] != "Easy":
        s += 3
    s += {"Easy": 0, "Medium": 1.0, "Hard": 0.5}[x["difficulty"]]
    if re.search(NEG_KW, t):
        s -= 8
    return s


def load():
    d = json.load(open(SRC, encoding="utf-8"))
    done_p = {norm(t) for t in DONE_PROBLEMS}
    done_m = {norm(t) for t in DONE_MATH}
    math = [x for x in d if x["section"] == "math"]
    probs = [x for x in d if x["section"] == "problems"]
    found_p = {norm(x["title"]) for x in probs} & done_p
    found_m = {norm(x["title"]) for x in math} & done_m
    miss = (done_p - found_p) | (done_m - found_m)
    labs = {x["id"]: x for x in d if x["section"] == "labs"}
    projs = {x["id"]: x for x in d if x["section"] == "projects"}
    return math, probs, done_p, done_m, miss, labs, projs


def build_pools(math, probs, done_p, done_m):
    M, n = [], len(math)
    for i, x in enumerate(math):
        if norm(x["title"]) in done_m:
            continue
        th = theme_of(x["title"], MATH_RULES, 10)
        M.append(dict(x, theme=th, order=-i, h=H["math"], diff="Medium"))
    seen, cand = set(), []
    for x in probs:
        k = norm(x["title"])
        if k in done_p or k in seen:
            continue
        c, t = x["category"], x["title"].lower()
        if c in MANDATORY:
            pass
        elif c == "Reinforcement Learning":
            if not re.search(RL_OK, t):
                continue
        elif c in EXTRA_CATS:
            if not re.search(CORE_KW, t) and not (c == "Linear Algebra" and x["difficulty"] != "Easy"):
                continue
        else:
            continue
        if re.search(NEG_KW, t):
            continue
        seen.add(k)
        cand.append(dict(x, theme=theme_of(x["title"], P_RULES, 8 if c in MANDATORY else 4),
                         order=int(x["id"]), h=H[x["difficulty"]], diff=x["difficulty"], pr=prio(x)))
    cand.sort(key=lambda x: (-x["pr"], x["order"]))
    return M, cand


LAB_BY_SAT = {
    date(2026, 10, 10): "mnist-build-neural-network-from-scratch-numpy-only",
    date(2026, 10, 17): "design-your-own-optimizer-numpy",
    date(2026, 10, 31): "design-your-own-normalization-layer",
    date(2026, 11, 7): "design-your-own-attention-mechanism",
    date(2026, 11, 21): "design-your-own-kv-cache-eviction-policy",
    date(2026, 12, 5): "design-your-own-quantizer",
    date(2026, 12, 26): "design-your-own-request-scheduler",
}
PROJ_BY_SAT = {
    date(2026, 10, 24): "build-your-own-teenygrad-a-tiny-tensor-autograd-engine",
    date(2026, 11, 14): "tiny-gpt-from-scratch",
    date(2026, 11, 28): "llm-inference-mechanics-prefill-decode-and-the-kv-cache",
    date(2026, 12, 12): "on-device-model-compression-pipeline",
    date(2026, 12, 19): "build-a-mini-llm-inference-server",
}
PROJ_FRI = {s - timedelta(days=1): (s, p) for s, p in PROJ_BY_SAT.items()}
LAB_THEME = {"mnist-build-neural-network-from-scratch-numpy-only": 3, "design-your-own-optimizer-numpy": 3,
             "design-your-own-normalization-layer": 4, "design-your-own-attention-mechanism": 5,
             "design-your-own-kv-cache-eviction-policy": 6, "design-your-own-quantizer": 7,
             "design-your-own-request-scheduler": 8}
PROJ_THEME = {"build-your-own-teenygrad-a-tiny-tensor-autograd-engine": 3, "tiny-gpt-from-scratch": 5,
              "llm-inference-mechanics-prefill-decode-and-the-kv-cache": 6, "on-device-model-compression-pipeline": 7,
              "build-a-mini-llm-inference-server": 8}


def weekday_slots():
    d, out = START, []
    while d <= END:
        if d.weekday() <= 4 and d not in PROJ_FRI:
            out.append(d)
        d += timedelta(days=1)
    return out


def pack(M, P, slots):
    M = sorted(M, key=lambda x: (x["theme"], x["order"]))
    P = sorted(P, key=lambda x: (x["theme"], RANK[x["diff"]], x["order"]))
    days = []
    for d in slots:
        if not M and not P:
            break
        cur = min([x["theme"] for x in (M[:1] + P[:1])])
        mi, pi, hrs = [], [], 0.0
        hard_idx = next((i for i, p in enumerate(P) if p["diff"] == "Hard" and p["theme" ] <= cur), None)
        non_hard_left = any(p["diff"] != "Hard" and p["theme"] <= cur for p in P)
        if hard_idx is not None and not non_hard_left:
            hp = P.pop(hard_idx)
            pi.append(hp); hrs += hp["h"]
            mid = next((i for i, p in enumerate(P) if p["diff"] == "Medium"), None)
            if mid is not None:
                q = P.pop(mid); pi.append(q); hrs += q["h"]
        else:
            while P and P[0]["diff"] != "Hard" and hrs + P[0]["h"] <= 2.4 and len(pi) < 4:
                q = P.pop(0); pi.append(q); hrs += q["h"]
            while M and hrs + M[0]["h"] <= DAY_MAX and (hrs < DAY_MIN or len(mi) < 2) and len(mi) < 8:
                q = M.pop(0); mi.append(q); hrs += q["h"]
            while P and P[0]["diff"] != "Hard" and hrs + P[0]["h"] <= DAY_MAX and hrs < DAY_MIN:
                q = P.pop(0); pi.append(q); hrs += q["h"]
            while M and hrs + M[0]["h"] <= DAY_MAX and hrs < DAY_MIN:
                q = M.pop(0); mi.append(q); hrs += q["h"]
        days.append((d, mi, pi, round(hrs, 2)))
    return days, M, P


def fit_problems(M, cand, slots):
    lo, hi, best = 0, len(cand), 0
    while lo <= hi:
        mid = (lo + hi) // 2
        _, rm, rp = pack([dict(x) for x in M], [dict(x) for x in cand[:mid]], slots)
        if not rm and not rp:
            best, lo = mid, mid + 1
        else:
            hi = mid - 1
    return best


def overall(items):
    r = [{"Easy": 1, "Medium": 2, "Hard": 3}[i["diff"]] for i in items]
    m = sum(r) / len(r)
    return "Easy" if m < 1.5 else ("Medium" if m < 2.4 else "Hard")


def main():
    math, probs, done_p, done_m, miss, labs, projs = load()
    if miss:
        print("CẢNH BÁO - không khớp được tên đã làm:", miss)
    M, cand = build_pools(math, probs, done_p, done_m)
    slots = weekday_slots()
    n = fit_problems(M, cand, slots)
    chosen = cand[:n]
    days, rm, rp = pack([dict(x) for x in M], [dict(x) for x in chosen], slots)
    print(f"math còn lại: {len(M)} | problems ứng viên: {len(cand)} | problems xếp được: {n} | ngày thường dùng: {len(days)}/{len(slots)}")

    rows, week_probs = {}, {}
    for d, mi, pi, hrs in days:
        items = mi + pi
        th = [x["theme"] for x in items]
        main_t = max(set(th), key=th.count)
        sec = ";".join(s for s, l in (("math", mi), ("problems", pi)) if l)
        note = THEMES[main_t][1]
        rows[d] = dict(Section=sec, Item_ID=";".join(x["id"] for x in items),
                       Task_Title="; ".join(x["title"].strip() for x in items),
                       Category=THEMES[main_t][0], Difficulty=overall(items),
                       Estimated_Hours=f"{hrs:.2f}".rstrip("0").rstrip("."), Focus_Note=note)
        week_probs.setdefault(d.isocalendar()[1], []).extend(pi)

    def review_for(sat):
        wk = sat.isocalendar()[1]
        pool = sorted(week_probs.get(wk, []), key=lambda x: -RANK[x["diff"]])[:2]
        return pool

    for d in [START + timedelta(days=i) for i in range((END - START).days + 1)]:
        if d in PROJ_FRI:
            s, pid = PROJ_FRI[d]
            p = projs[pid]
            rows[d] = dict(Section="projects", Item_ID=pid, Task_Title=f"{p['title']} (Part 1/2: dựng khung & core)",
                           Category=THEMES[PROJ_THEME[pid]][0], Difficulty="Hard", Estimated_Hours="3.75",
                           Focus_Note="Tự tay dựng thành phần cốt lõi from scratch - " + THEMES[PROJ_THEME[pid]][1].lower())
        elif d in PROJ_BY_SAT:
            pid = PROJ_BY_SAT[d]; p = projs[pid]; rv = review_for(d)
            rows[d] = dict(Section="projects", Item_ID=";".join([pid] + [x["id"] for x in rv]),
                           Task_Title="; ".join([f"{p['title']} (Part 2/2: hoàn thiện & kiểm thử)"] + [f"[Review] {x['title'].strip()}" for x in rv]),
                           Category=THEMES[PROJ_THEME[pid]][0], Difficulty="Hard", Estimated_Hours="3.75",
                           Focus_Note="Hoàn tất project + làm lại 2 bài khó nhất tuần để chốt kiến thức - " + THEMES[PROJ_THEME[pid]][1].lower())
        elif d in LAB_BY_SAT:
            lid = LAB_BY_SAT[d]; l = labs[lid]; rv = review_for(d)
            rows[d] = dict(Section="labs", Item_ID=";".join([lid] + [x["id"] for x in rv]),
                           Task_Title="; ".join([l["title"]] + [f"[Review] {x['title'].strip()}" for x in rv]),
                           Category=THEMES[LAB_THEME[lid]][0], Difficulty="Medium", Estimated_Hours="3.75",
                           Focus_Note="Lab tổng hợp cuối tuần + review 2 bài khó - " + THEMES[LAB_THEME[lid]][1].lower())
        elif d.weekday() == 6:
            rows[d] = dict(Section="", Item_ID="", Task_Title="Rest Day - nghỉ hoàn toàn", Category="Rest",
                           Difficulty="", Estimated_Hours="0", Focus_Note="Nghỉ ngơi để não củng cố trí nhớ và tránh burn-out.")
        elif d not in rows:
            rows[d] = dict(Section="", Item_ID="", Task_Title="Buffer / ôn tập tự do (đã hết bài trong kế hoạch)",
                           Category="Review", Difficulty="Easy", Estimated_Hours="3.5",
                           Focus_Note="Ngày đệm để bù bài trễ hoặc làm lại bài khó.")
    cols = ["Date", "DayOfWeek", "Type", "Section", "Item_ID", "Task_Title", "Category", "Difficulty", "Estimated_Hours", "Focus_Note"]
    with open(OUT, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        d = START
        while d <= END:
            r = rows[d]
            w.writerow(dict(Date=d.isoformat(), DayOfWeek=d.strftime("%A"), Type="Rest" if d.weekday() == 6 else "Study", **r))
            d += timedelta(days=1)
    print("Đã ghi", OUT)
    from google.colab import files
    files.download(OUT)


if __name__ == "__main__":
    main()


math còn lại: 200 | problems ứng viên: 437 | problems xếp được: 115 | ngày thường dùng: 57/57
Đã ghi learning_schedule_deepml_2026.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>